# Reproducing *Fail Fast* (arXiv:2511.15177) for BB(6) = [[72,12,6]]: Table 2, Figures 9 & 10

This notebook reproduces the paper's **BB(6)** results — the **Table 2** min-weight properties, the
**Figure 10** Relay-BP LER curve, and the **Figure 9** splitting cross-check — using all three
"fail-fast" techniques:

| Technique | What it gives | Status here |
|---|---|---|
| **I** — failure-spectrum ansatz + importance sampling | the LER(p) curve | reproduced (f3 **and** f5) |
| **II** — exact min-weight onset (D, w0, f0, \|L(D)\|, \|F\|) | the onset that pins the ansatz | **reproduced exactly** |
| **III** — replica-exchange splitting | independent cross-check | agrees across near-threshold range |

**Architecture:** a **Stim circuit** is the source; `single_sector_dem(circuit, sector=0)` projects it
to the paper's single-CSS-sector (Z-type) representation; Relay-BP and the importance sampler run on
those derived matrices. Changing the error model = rebuild the circuit; everything downstream follows.

All heavy computation was done by the pipeline scripts (`bb6_fig10_sweep.py`,
`bb6_reference_port.py`, `bb6_exact_enum_mitm.py`, `split_crosscheck.py`); this notebook loads the
saved results and re-fits/re-plots them (fast).

### Setup — imports, then load + fit (two cells)

The first cell only imports (fast). The **second** cell calls `bb6_report.compute()`, which fits the
f3 and f5 ansatze and bootstraps their confidence bands — this takes **~10-15 s** (the f5 fits
dominate), so the spinner on that cell is expected. It is *not* the imports hanging.

In [ ]:
import sys, pathlib
print("kernel python:", sys.executable)   # should be the 'qec' env; this report needs numpy/scipy/matplotlib
# Find this notebook's folder (it contains bb6_report.py) regardless of the kernel's working
# directory — Jupyter uses the notebook dir, but VS Code may use the workspace root.
_here = pathlib.Path.cwd()
_cands = [_here, *_here.parents] + [c / "notebooks" / "bb_code" for c in [_here, *_here.parents]]
_nbdir = next((c for c in _cands if (c / "bb6_report.py").exists()), None)
assert _nbdir is not None, "Could not locate bb6_report.py — run this notebook from inside the repo."
sys.path.insert(0, str(_nbdir)); sys.path.insert(0, str(_nbdir.parent.parent / "src"))
import numpy as np, matplotlib.pyplot as plt
import bb6_report                      # tested analysis module (this folder)
print("imports OK — run the next cell to load + fit (~10-15s)")

In [ ]:
import time
print("fitting f3 & f5 ansatze + bootstrap bands (~10-15s)...", flush=True)
_t = time.time()
R = bb6_report.compute()              # loads bb6_fig10_curve/, fits f3 & f5 ansatze, bootstraps bands
print(f"loaded from {_nbdir.name} in {time.time()-_t:.0f}s — "
      f"{R['isp'].size} p-grid points, {R['w'].size} sampled weights")

## The system

- **Code:** bivariate-bicycle BB(6) = [[72,12,6]] (l=m=6, A=x³+y+y², B=y³+x+x²).
- **Circuit:** Bravyi depth-7 syndrome schedule, d=6 noisy cycles + 2 noiseless cycles
  (`bb6_reference_port.py` matches the reference exactly; the Stim builder in
  `bb6_fig10_sweep.py` is within 0.08%).
- **Noise:** standard circuit-level depolarizing (CNOT `DEPOLARIZE2(p)`, idle `DEPOLARIZE1(p)`,
  prep/meas flip `p`); base rate q = p/15.
- **Decoder:** Relay-BP (paper §2.4 settings: γ₀=0.125, 80+600×60 iters, S=6).
- **Representation:** single Z-sector. N (expanded) = 46,224 fault mechanisms.

## Technique II — exact min-weight onset (Table 2)

The onset is found by an exhaustive weight-6 logical enumeration (`bb6_exact_enum_mitm.py`):
canonical-detector anchoring + GF(2)-linear-hash meet-in-the-middle, expanded by the 36 Z₆×Z₆
toric shifts. BP-OSD heuristics plateau at \|L(D)\|=1452; the exact search finds the complete **1524**.

In [ ]:
E, P = bb6_report.EXACT, bb6_report.PAPER
rows = [("N~ (compressed)", E["n_compressed"], 2233),
        ("N  (expanded)",   E["n_expanded"],   P["n_expanded"]),
        ("D",               E["D"],            6),
        ("|L(D)| compressed", E["L_compressed"], 1524),
        ("|L(D)| expanded", f"{E['L_expanded']:.3e}", f"{P['L_expanded']:.3e}"),
        ("|F(D/2)|",        f"{E['F']:,}",     f"{P['F']:.3e}"),
        ("f0 = f*(D/2)",    f"{E['f0']:.4g}",  f"{P['f0']:.3g}")]
print(f"{'quantity':<20}{'ours (exact)':>18}{'paper':>16}")
for name, ours, paper in rows:
    print(f"{name:<20}{str(ours):>18}{str(paper):>16}")

## Technique I — failure spectrum and ansatz

At each fault weight `w`, importance sampling draws random weight-`w` fault patterns, decodes with
Relay-BP, and records the failure fraction `f(w)`. The spectrum is fit by the `f3` and `f5` ansatze,
**pinned** at the exact onset (w0=3, f0=2.324×10⁻⁵), then reweighted to LER(p):

LER(p) = Σ_w C(N,w) qʷ (1−q)^(N−w) f(w),  q = (p/15)(p/p_ref).

In [ ]:
for m in ("f3", "f5"):
    pars = {k: round(float(v), 4) for k, v in R["fits"][m].params.items()}
    ler = R["LER"][m][np.argmin(abs(R["p_grid"] - 1e-4))]
    print(f"{m}: params={pars}   LER(1e-4) = {ler:.3e}")
print("\nf3 and f5 agree to ~1% -> the low-p extrapolation is robust to ansatz choice.")

### Weight-space view: failure spectrum f(w) (with the exact Technique-II onset point)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
bb6_report.fig_failure_spectrum(R, ax)
plt.show()

### Figure 10: logical vs physical error rate (all three techniques)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
bb6_report.fig_ler_vs_p(R, ax)
plt.show()
print("LER(1e-4): f3 = %.2e, f5 = %.2e   (paper Fig. 10 ~ 1e-7)"
      % (R["LER"]["f3"][np.argmin(abs(R["p_grid"]-1e-4))],
         R["LER"]["f5"][np.argmin(abs(R["p_grid"]-1e-4))]))

## Technique III — Metropolis splitting cross-check (replica exchange)

Splitting (`split_crosscheck.py`) runs on the **same** single-sector representation and walks a
geometric ladder of physical rates from near threshold **down to deep sub-threshold (6×10⁻³ →
10⁻⁴, 48 levels)**, estimating each rung's conditional reweight ratio so it never has to sample the
rare onset directly — the regime where direct IS collapses (see below).

The naive single-flip chain is biased: each ladder reweight term depends only on the config *weight*
and is a decreasing function of it, so a chain trapped near its seed weight is biased — min-weight
seeds **overshoot**, high-weight seeds **undershoot**. The fix is **replica exchange** with a
**balanced add/remove proposal** (`replica_exchange_estimate`): the balanced proposal mixes weight
in O(steps), and swaps between adjacent rates let configs migrate to their equilibrium weight at
each level.

**Result of the deep run:** the replica-exchange estimate tracks the Technique-I ansatz at a steady
**0.99–1.26×** across the *entire* 6×10⁻³ → 10⁻⁴ range — at 10⁻⁴ it gives **1.42×10⁻⁷** vs the
onset-pinned ansatz **1.16×10⁻⁷**, an independent confirmation (using *no* exact-onset input) exactly
where direct IS undershoots ~50×. The chain **mean weight descends monotonically 22 → 3.4** (mixing
into the onset region w₀=3) with **swap acceptance 0.81–0.98**. The small steady ~1.2× offset is
consistent with the lighter cross-check decoder (num_sets=100 vs the curve's 600 → slightly more
failures); it is a constant offset, not a drift, so it is not a mixing artefact.

The two one-sided sequential runs (min-weight-seed = over, MC-seed = under) form a **bracket** — a
mixing check that works near threshold but **collapses below ~8×10⁻⁴** (the very trapping replica
exchange cures), so the bracket band is shown only where it genuinely brackets.

In [ ]:
if R["split"]:
    c = bb6_report.splitting_comparison(R)   # de-aliased: ansatz evaluated at each rung's exact p
    print(f"{'p':>10}{'tempered (SE)':>20}{'Tech-I':>11}{'ratio':>7}  agree  bracket")
    for k in range(0, c['p'].size, 2):       # every other rung (49 total)
        ag = ' ok ' if c['valid'][k] else '>2x '
        br = 'inside' if c['inside_bracket'][k] else 'collapsed'
        print(f"{c['p'][k]:>10.2e}{c['tP'][k]:>11.2e} (±{c['tSE'][k]:.0e}){c['ansatz'][k]:>11.2e}"
              f"{c['ratio'][k]:>7.2f}  {ag}  {br}")
    sa = R["split"]["diagnostics"]["swap_accept"]; mw = R["split"]["diagnostics"]["mean_weight"]
    print(f"\nratio range {c['ratio'].min():.2f}-{c['ratio'].max():.2f} over {c['p'].size} rungs; "
          f"swap-accept {min(sa):.2f}..{max(sa):.2f}; mean weight {mw[0]:.1f} (hi-q) -> {mw[-1]:.1f} (lo-q)")

### Cross-check against paper Figure 9 (BB(6)-relay)

Figure 9 of the paper shows multi-seeded splitting for BB(6): panel **(a)** LER vs p (downward
splitting, upward splitting, and Monte Carlo) and panel **(c)** the *weight distribution of failing
configurations* vs p. Our analogs below:

- **(a)** the replica-exchange splitting points lie on the ansatz / IS curve across the **full
  6×10⁻³ → 10⁻⁴ range** — like the paper's *downward splitting* + Monte Carlo.
- **(c)** the failing-config weight rises from **~3.4 just above the onset (w₀=3) at p=10⁻⁴ to ~22
  near threshold**; our chain mean weights track the analytic π_q(w) median (from the measured
  f(w)) — matching the shape of the paper's BB(6) weight distributions.

The paper's *upward/downward* splitting is our *over/under* bracket, and it explicitly notes the
upward splitting "struggle[s] to converge or fully mix" — exactly our overshoot. The downward (our
replica-exchange) estimate is the reliable one.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 5.8))
bb6_report.fig_ler_vs_p(R, axL); axL.set_title("(a) LER vs $p$  (cf. paper Fig. 9a)")
bb6_report.fig_weight_vs_p(R, axR)
plt.show()

## Convergence & error bars — can each technique have an error bar?

| Technique | Error bar? | Source |
|---|---|---|
| **I — IS points** | **Yes** | binomial SE per weight √(f(1−f)/T), propagated through the reweighting |
| **I — ansatz** | **Yes** | 90% bootstrap band (resample failures, refit, B=300); came out tight |
| **II — onset (w0, f0)** | **No — exact** | exhaustive enumeration; \|L(D)\|=1524, \|F\|=3.825×10⁸ are exact counts |
| **III — replica-exchange splitting** | **Yes** | across-walker SE; the under/over bracket independently checks mixing |

**Convergence:** the replica-exchange splitting now agrees with Technique I across the *entire*
near-threshold ladder (p≈6×10⁻³→3×10⁻³, ratios ≈0.7–1.4 with no systematic bias), validated by
lying inside the under/over bracket; its per-level mean weight matches the analytic target. IS and
the ansatze agree across the whole sampled range; f3 and f5 agree to ~1% in the deep extrapolation.
The bootstrap band captures *statistical* uncertainty given a fixed ansatz — the f3-vs-f5 spread is
the (small) *model* uncertainty. Technique I remains the authoritative low-p curve.

## Summary

- **Table 2 reproduced exactly** (single-sector reference circuit): N=46,224, Ñ=2233, D=6,
  \|L(D)\|=1524, \|L(D)\|ₑₓₚ=6.01×10¹², \|F\|=3.825×10⁸→3.83, f0=2.324×10⁻⁵.
- **Figure 10 reproduced**: f3 and f5 ansatze both give **LER(10⁻⁴) ≈ 1.2×10⁻⁷** (paper ~10⁻⁷).
- **All three techniques agree** where comparable; Technique-I is the authoritative low-p curve,
  Technique-II provides the exact onset anchor, Technique-III confirms at threshold.